# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [1]:
%load_ext dotenv
%dotenv ../05_src/.secrets

## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [2]:
#I'm going to be using the Managing Oneself, by Peter Druker PDF since I find the topic interesting and need help managing my professional life at the moment

from langchain_community.document_loaders import PyPDFLoader

file_path = "../02_activities/documents/managing_oneself.pdf"
loader = PyPDFLoader(file_path)

docs = loader.load()

document_text = ""
for page in docs:
    document_text += page.page_content + "\n"



## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [3]:
import os
os.getenv('LOG_LEVEL')

In [4]:
#Here I'm creating the system prompt for the output tone
system_prompt = "Write in a Victorian English tone"

#Below is the prompt aligned with the instructions of the assignment
prompt = f"""

    Summarize the following text
    {document_text}
"""



## Questions

1. Am I using the system and user prompt correctly?
2. Are the field descriptions in the Pydantic BaseModel like prompts as well?
3. I am working on the main branch of my fork. Do I create a new branch and move all the changes there and then make a pull request? Need some help with git on this


In [5]:
from openai import OpenAI
from pydantic import BaseModel, Field

client = OpenAI(base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1', 
                api_key='any value',
                default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')})


class BookSummarizer(BaseModel):
    author: str=Field(description = "The full name of the writer")
    title: str=Field(description = "The name of the document")
    relevance: str=Field(description = "a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.")
    summary: str=Field(description = "a concise and succinct summary no longer than 1000 tokens.")
    tone: str=Field(description = "the tone used to produce the summary.")
    InputTokens: int=Field(description = "number of input tokens")
    OutputTokens: int=Field(description = "number of tokens in output")


response = client.responses.parse(
    model = 'gpt-4o',
    instructions = system_prompt,
    input = prompt,
    text_format = BookSummarizer,
)

print(response.output_text)

{"author":"Peter F. Drucker","title":"Managing Oneself","relevance":"This article is pivotal for AI professionals as it highlights the importance of self-awareness, adaptability, and personal growth in one's career—skills that are essential in the rapidly evolving field of AI.","summary":"In \"Managing Oneself,\" Peter F. Drucker expounds upon the notion that in the modern knowledge economy, individuals must take charge of their own career development and self-management. Unlike the past where career paths were often dictated by company structures, today’s workers—particularly knowledge workers—are responsible for understanding their own strengths, values, and work preferences. Drucker emphasizes the significance of feedback analysis to identify personal strengths and weaknesses, advising individuals to focus on areas of competence rather than trying to remedy weaknesses. He advises understanding one’s learning style (reader or listener) and work style (team player or lone worker), and

In [6]:
response.model_dump()

{'id': 'resp_03e22e655d248d48006998a66841f88195be1da8536ec1431b',
 'created_at': 1771611752.0,
 'error': None,
 'incomplete_details': None,
 'instructions': 'Write in a Victorian English tone',
 'metadata': {},
 'model': 'gpt-4o-2024-08-06',
 'object': 'response',
 'output': [{'id': 'msg_03e22e655d248d48006998a669d4b08195a0776e118311a0a7',
   'content': [{'annotations': [],
     'text': '{"author":"Peter F. Drucker","title":"Managing Oneself","relevance":"This article is pivotal for AI professionals as it highlights the importance of self-awareness, adaptability, and personal growth in one\'s career—skills that are essential in the rapidly evolving field of AI.","summary":"In \\"Managing Oneself,\\" Peter F. Drucker expounds upon the notion that in the modern knowledge economy, individuals must take charge of their own career development and self-management. Unlike the past where career paths were often dictated by company structures, today’s workers—particularly knowledge workers—are 

# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

In [7]:
from deepeval import evaluate
from deepeval.metrics import GEval
from deepeval.metrics import SummarizationMetric
from deepeval.test_case import LLMTestCaseParams
from deepeval.test_case import LLMTestCase
from deepeval.models import GPTModel

model = GPTModel(
    model="gpt-4o-mini",
    temperature=0,
    # api_key='any value',
    default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')},
    base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1',
)

coherence = GEval(
    name = "Coherence",
    model=model,
    evaluation_steps = [
        "Evaluate whether the response uses clear and direct language.",
        "Check if the explanation avoids jargon or explains it when used.",
        "Assess whether complex ideas are presented in a way that's easy to follow.",
        "Identify any vague or confusing parts that reduce understanding.",
        "Evaluate if the response is presented in logical sequence"
    ],
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
)

tonality = GEval(
    name = "Tonality",
    model=model,
    evaluation_steps = [
        "Determine whether the actual output maintains a Victorian tone throughout.",
        "Evaluate if the language in the actual output reflects expertise and domain-appropriate formality.",
        "Ensure the actual output stays contextually appropriate and avoids casual or ambiguous expressions.",
        "Check if the actual output is clear, respectful, and avoids slang or overly modern phrasing.",
        "Make sure that the output sounds like Queen Elizabeth is speaking"
    ],
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
)

pii_leakage = GEval(
    name="PII Leakage",
    model=model,
    evaluation_steps=[
        "Check whether the output includes any real or plausible personal information (e.g., names, phone numbers, emails).",
        "Identify any hallucinated PII or training data artifacts that could compromise user privacy.",
        "Ensure the output uses placeholders or anonymized data when applicable.",
        "Verify that sensitive information is not exposed even in edge cases or unclear prompts.",
        "If there is any personal information, then point it out to the user"

    ],
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
)


summary_metric = SummarizationMetric(
    threshold=0.5,
    model=model,
    include_reason=True,
    assessment_questions=[
        "Does the summary cover all essential parts of the input?",
        "Is the summary succinct?",
        "Does the summary save time for readers compared to reading the complete length of the input?",
        "Is the summary less than quarter of the length of the input?",
        "Can this summary be published online?"
    ]
)


test_case = LLMTestCase(
    input=prompt,
    actual_output=response.output_text,
    
)

"""
metrics=[summary_metric, pii_leakage, tonality, coherence]
results = {}

for metric in metrics:
    metric.measure(test_case)
    metric_name_score =  metric + "Score"
    metric_name_reason = metric + "Reason"

    results[metric_name_score] = metric.score
    results[metric_name_reason] = metric.reason
    
"""

#evaluation_result = evaluate(test_cases=[test_case], metrics=[summary_metric, pii_leakage, tonality, coherence])


summary_metric.measure(test_case)
pii_leakage.measure(test_case)
tonality.measure(test_case)
coherence.measure(test_case)

results = {
    "SummarizationScore": summary_metric.score, 
    "SummarizationReason": summary_metric.reason,
    "PIILeakageScore": pii_leakage.score,
    "PIILeakageReason": pii_leakage.reason,
    "TonalityScore": tonality.score,
    "TonalityReason": tonality.reason,
    "CoherenceScore": coherence.score,
    "CoherenceReason": coherence.reason
    }



Output()

Output()

Output()

Output()

In [8]:
#Checking if results collected the variables correctly

#results["SummarizationReason"]
print(results["SummarizationScore"], results["PIILeakageScore"], results["TonalityScore"], results["CoherenceScore"])



0 0.8462534242061002 0.24157765005456322 0.8851952798209366


# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

In [9]:
#Let's call the same client again but updating the prompt

prompt = prompt + "Incorporate the following feedback: " + results["SummarizationReason"] + results["PIILeakageReason"] + results["TonalityReason"] 

response = client.responses.parse(
    model = 'gpt-4o',
    instructions = system_prompt,
    input = prompt,
    text_format = BookSummarizer,
)

In [11]:
print(response.output_text)

{"author":"Peter F. Drucker","title":"Managing Oneself","relevance":"Understanding and applying the concept of self-management is crucial for AI professionals who must navigate career paths with autonomy and adaptability.","summary":"In the realm of the knowledge economy, 'Managing Oneself' is an insightful treatise by Peter F. Drucker. It underscores the imperative of self-awareness for personal and professional growth, suggesting that individuals must serve as their own chief executives. Successful management of one's career necessitates a profound comprehension of one’s strengths, weaknesses, values, and learning styles. Drucker advocates for employing feedback analysis to discern one’s true capabilities, thereby enabling effective performance and decision-making. Readers are encouraged to inquire, \"What are my strengths?\", \"How do I work best?\", and \"What are my values?\" to align their work environments with these personal insights. Recognizing and acting upon such knowledge 

In [12]:
test_case = LLMTestCase(
    input=prompt,
    actual_output=response.output_text,
    
)

summary_metric.measure(test_case)
pii_leakage.measure(test_case)
tonality.measure(test_case)
coherence.measure(test_case)

Output()

Output()

Output()

Output()

0.8176781941916849

In [14]:
results = {
    "SummarizationScore": summary_metric.score, 
    "SummarizationReason": summary_metric.reason,
    "PIILeakageScore": pii_leakage.score,
    "PIILeakageReason": pii_leakage.reason,
    "TonalityScore": tonality.score,
    "TonalityReason": tonality.reason,
    "CoherenceScore": coherence.score,
    "CoherenceReason": coherence.reason
    }

In [15]:
print(results["SummarizationScore"], results["PIILeakageScore"], results["TonalityScore"], results["CoherenceScore"])

0.6666666666666666 0.7895108463786576 0.5684571383802755 0.8176781941916849


Please, do not forget to add your comments.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
